In [0]:
%sql
select * from zara_lakehouse.retail_bronze.product_information_bronze

In [0]:
%sql
select count (*) from zara_lakehouse.retail_bronze.product_information_bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df = spark.sql("select * from zara_lakehouse.retail_bronze.product_information_bronze")

In [0]:
duplicates_df = df.groupBy("Product ID") \
    .count() \
    .filter("count > 1")

In [0]:
display(duplicates_df)

In [0]:
df_null = df.withColumn("is_id_null", F.col("product_id").isNull())
display(df_null)


In [0]:
df_null_count = df_null.groupBy("is_id_null").count().show()

In [0]:
summary_df = df_null.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("is_id_null") == True, 1).otherwise(0)).alias("null_rows"),
    F.sum(F.when(F.col("is_id_null") == False, 1).otherwise(0)).alias("non_null_rows")
).withColumn(
    "ingestion_time", F.current_timestamp()
)

display (summary_df)